In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from collections import deque
import pandas as pd
import csv
import os

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

# angle calculation
def calculate_angle(point_a, point_b, point_c):
    point_a, point_b, point_c = np.array(point_a), np.array(point_b), np.array(point_c)
    vector_ba = point_a - point_b
    vector_bc = point_c - point_b
    cosine_angle = np.dot(vector_ba, vector_bc) / (np.linalg.norm(vector_ba) * np.linalg.norm(vector_bc))
    angle_in_degrees = np.degrees(np.arccos(np.clip(cosine_angle, -1.0, 1.0)))
    return angle_in_degrees

# ml feature preparation
def prepare_ml_features(lm, side, frame_width, frame_height):
    if side == "Right":
        hip, knee, ankle = 24, 26, 28
        shoulder, elbow, wrist = 12, 14, 16
    else:
        hip, knee, ankle = 23, 25, 27
        shoulder, elbow, wrist = 11, 13, 15

    # normalization and torso scaling (front view)
    mid_hip_x = (lm[24].x + lm[23].x) / 2
    mid_hip_y = (lm[24].y + lm[23].y) / 2
    mid_shoulder_x = (lm[12].x + lm[11].x) / 2
    mid_shoulder_y = (lm[12].y + lm[11].y) / 2
    
    torso_dist = abs(mid_shoulder_y - mid_hip_y)
    if torso_dist == 0: torso_dist = 1
    
    def norm_x(l_idx): return (lm[l_idx].x - mid_hip_x) / torso_dist
    def norm_y(l_idx): return (lm[l_idx].y - mid_hip_y) / torso_dist

    # feature calculation
    features = {
        'n_ankle_x': round(norm_x(ankle), 4),
        'n_ankle_y': round(norm_y(ankle), 4),
        'n_knee_x': round(norm_x(knee), 4),
        'n_knee_y': round(norm_y(knee), 4),
        'n_knee_to_hip_x': round(norm_x(knee) - norm_x(hip), 4),
        'n_ankle_to_hip_x': round(norm_x(ankle) - norm_x(hip), 4),
        'n_shoulder_x': round(norm_x(shoulder), 4),
        'n_elbow_x': round(norm_x(elbow), 4),
        'n_wrist_x': round(norm_x(wrist), 4)
    }
    return features




# video features extraction
def analyze_video(video_path):
    video_capture = cv2.VideoCapture(video_path)
    frames_per_second = video_capture.get(cv2.CAP_PROP_FPS)

    # thresholds and setup
    push_off_threshold = 0.05
    min_swing_frames = frames_per_second * 0.22
    min_contact_frames = 5
    gct_timeout = 1.0

    all_step_metrics_storage = []

    history_window_size = 9
    right_ankle_y_history = deque(maxlen=history_window_size)
    left_ankle_y_history = deque(maxlen=history_window_size)

    cadence_history = deque(maxlen=5)
    gct_filter_history = deque(maxlen=5)

    leg_is_on_ground = {"Right": False, "Left": False}
    ankle_y_at_contact = {"Right": None, "Left": None}

    last_leg_that_landed = None
    last_strike_frame_index = 0
    previous_strike_frame = None
    avg_cadence = 0
    right_step_count = 0
    left_step_count = 0
    current_status_event = ""

    with mp_pose.Pose(min_detection_confidence=0.7, min_tracking_confidence=0.7) as pose_analyzer:
        while video_capture.isOpened():
            ret, frame = video_capture.read()
            if not ret:
                break
    
            overlay_layer = frame.copy()
            display_frame = frame.copy()
            frame_height, frame_width, _ = frame.shape
            results = pose_analyzer.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    
            if results.pose_landmarks:
                landmarks = results.pose_landmarks.landmark
    
                def get_pixel_point(idx):
                    return np.array([
                        int(landmarks[idx].x * frame_width),
                        int(landmarks[idx].y * frame_height)
                    ])
    
                # keypoint extraction
                right_hip_px, right_knee_px, right_ankle_px = get_pixel_point(24), get_pixel_point(26), get_pixel_point(28)
                left_hip_px, left_knee_px, left_ankle_px = get_pixel_point(23), get_pixel_point(25), get_pixel_point(27)
    
                right_shoulder_px, right_elbow_px, right_wrist_px = get_pixel_point(12), get_pixel_point(14), get_pixel_point(16)
                left_shoulder_px, left_elbow_px, left_wrist_px = get_pixel_point(11), get_pixel_point(13), get_pixel_point(15)
    
                nose_px = get_pixel_point(0)
    
                L_hip = np.array([landmarks[23].x, landmarks[23].y])
                R_hip = np.array([landmarks[24].x, landmarks[24].y])
    
                L_sh = np.array([landmarks[11].x, landmarks[11].y])
                R_sh = np.array([landmarks[12].x, landmarks[12].y])
    
                L_ank = np.array([landmarks[27].x, landmarks[27].y])
                R_ank = np.array([landmarks[28].x, landmarks[28].y])
    
                L_wri = np.array([landmarks[15].x, landmarks[15].y])
                R_wri = np.array([landmarks[16].x, landmarks[16].y])
    
                nose = np.array([landmarks[0].x, landmarks[0].y])
    
                torso_height = abs(((L_hip[1] + R_hip[1]) / 2) - ((L_sh[1] + R_sh[1]) / 2))
                mid_hip_x = (L_hip[0] + R_hip[0]) / 2
                hip_width = abs(R_hip[0] - L_hip[0]) + 1e-9
    
                # --- front-view metrike ---
                dx_hip = abs(R_hip[0] - L_hip[0])
                dy_hip = abs(R_hip[1] - L_hip[1])
                hip_drop_value_deg = np.degrees(np.arctan2(dy_hip, dx_hip + 1e-9))
    
                dx_sh = abs(R_sh[0] - L_sh[0])
                dy_sh = abs(R_sh[1] - L_sh[1])
                shoulder_drop_value_deg = np.degrees(np.arctan2(dy_sh, dx_sh + 1e-9))
    
                right_foot_offset_rel = (R_ank[0] - mid_hip_x) / hip_width
                left_foot_offset_rel = (L_ank[0] - mid_hip_x) / hip_width
    
                dx_r = abs(landmarks[16].x - landmarks[12].x)
                dy_r = abs(landmarks[16].y - landmarks[12].y)
                right_arm_inward_deg = np.degrees(np.arctan2(dx_r, dy_r + 1e-9))
    
                dx_l = abs(landmarks[15].x - landmarks[11].x)
                dy_l = abs(landmarks[15].y - landmarks[11].y)
                left_arm_inward_deg = np.degrees(np.arctan2(dx_l, dy_l + 1e-9))
    
                arm_inward_deg = (right_arm_inward_deg + left_arm_inward_deg) / 2
    
                right_fist_height_rel = (nose[1] - R_wri[1]) / (torso_height + 1e-9)
                left_fist_height_rel = (nose[1] - L_wri[1]) / (torso_height + 1e-9)
    
                #drawing and overlay
                torso_pts = np.array([right_shoulder_px, left_shoulder_px, left_hip_px, right_hip_px], np.int32)
                cv2.fillPoly(overlay_layer, [torso_pts], (0, 255, 0))
                cv2.addWeighted(overlay_layer, 0.15, display_frame, 0.85, 0, display_frame)
                cv2.polylines(display_frame, [torso_pts], True, (255, 255, 255), 2)
                cv2.line(display_frame, tuple(right_shoulder_px), tuple(left_hip_px), (255, 255, 255), 1)
                cv2.line(display_frame, tuple(left_shoulder_px), tuple(right_hip_px), (255, 255, 255), 1)
                cv2.line(display_frame, tuple(right_ankle_px), tuple(left_ankle_px), (255, 0, 255), 2)
    
                for h, k, a, c in [
                    (right_hip_px, right_knee_px, right_ankle_px, (0, 255, 0)),
                    (left_hip_px, left_knee_px, left_ankle_px, (0, 255, 255))
                ]:
                    cv2.line(display_frame, tuple(h), tuple(k), c, 3)
                    cv2.line(display_frame, tuple(k), tuple(a), c, 3)
    
                for s, e, w in [
                    (right_shoulder_px, right_elbow_px, right_wrist_px),
                    (left_shoulder_px, left_elbow_px, left_wrist_px)
                ]:
                    cv2.line(display_frame, tuple(s), tuple(e), (255, 165, 0), 3)
                    cv2.line(display_frame, tuple(e), tuple(w), (255, 165, 0), 3)
    
                # --- histories (strike detekcija) ---
                right_ankle_y_history.append(landmarks[28].y)
                left_ankle_y_history.append(landmarks[27].y)
    
                current_frame = int(video_capture.get(cv2.CAP_PROP_POS_FRAMES))
    
                # strike detection logic
                if len(right_ankle_y_history) == history_window_size and len(left_ankle_y_history) == history_window_size:
                    middle_index = history_window_size // 2
    
                    for side, history, ankle_y in [
                        ("Right", right_ankle_y_history, landmarks[28].y),
                        ("Left",  left_ankle_y_history,  landmarks[27].y)
                    ]:
                        if (
                            last_leg_that_landed != side and
                            history[middle_index] == max(history) and
                            (current_frame - last_strike_frame_index) > min_swing_frames
                        ):
                            leg_is_on_ground[side] = True
                            ankle_y_at_contact[side] = ankle_y
                            last_leg_that_landed = side
                            current_status_event = f"{side.upper()} STRIKE"
    
                            if previous_strike_frame is not None:
                                frames_between = current_frame - previous_strike_frame
                                if frames_between > 0:
                                    cadence_history.append((60 * frames_per_second) / frames_between)
                                    avg_cadence = sum(cadence_history) / len(cadence_history)
    
                            previous_strike_frame = current_frame
                            last_strike_frame_index = current_frame
    
                            if side == "Right":
                                right_step_count += 1
                                foot_now = right_foot_offset_rel
                                fist_now = right_fist_height_rel
                            else:
                                left_step_count += 1
                                foot_now = left_foot_offset_rel
                                fist_now = left_fist_height_rel
    
                            ml_data = prepare_ml_features(landmarks, side, frame_width, frame_height)
    
                            all_step_metrics_storage.append({
                                "side": side,
                                "start_frame": current_frame,
                                "cadence": avg_cadence,
                                "done": False,
                                "gct": 0,
    
                                "hip_drop_max_deg": hip_drop_value_deg,
                                "shoulder_drop_max_deg": shoulder_drop_value_deg,
    
                                "arm_inward_sum": arm_inward_deg,
                                "fist_height_sum": fist_now,
                                "foot_offset_sum": foot_now,
                                "n": 1,
    
                                **ml_data
                            })
    
                #PUSH-OFF and GCT
                for side, current_y in [("Right", landmarks[28].y), ("Left", landmarks[27].y)]:
                    if leg_is_on_ground[side]:
                        for rec in reversed(all_step_metrics_storage):
                            if rec["side"] == side and not rec["done"]:
                                if hip_drop_value_deg > rec["hip_drop_max_deg"]:
                                    rec["hip_drop_max_deg"] = hip_drop_value_deg
                                if shoulder_drop_value_deg > rec["shoulder_drop_max_deg"]:
                                    rec["shoulder_drop_max_deg"] = shoulder_drop_value_deg
    
                                rec["arm_inward_sum"] += arm_inward_deg
                                rec["fist_height_sum"] += (right_fist_height_rel if side == "Right" else left_fist_height_rel)
                                rec["foot_offset_sum"] += (right_foot_offset_rel if side == "Right" else left_foot_offset_rel)
                                rec["n"] += 1
    
                                frames_on_ground = current_frame - rec["start_frame"]
    
                                if (frames_on_ground / frames_per_second) > gct_timeout:
                                    leg_is_on_ground[side] = False
                                    rec["done"] = True
                                    rec["gct"] = 0
                                    break
    
                                if (ankle_y_at_contact[side] - current_y) > push_off_threshold and frames_on_ground >= min_contact_frames:
                                    raw_val = (frames_on_ground / frames_per_second) * 1000
                                    gct_filter_history.append(raw_val)
                                    rec["gct"] = sum(gct_filter_history) / len(gct_filter_history) if len(gct_filter_history) > 0 else raw_val
    
                                    n = max(1, rec["n"])
                                    rec["arm_inward_mean_deg"] = rec["arm_inward_sum"] / n
                                    rec["fist_height_mean_rel"] = rec["fist_height_sum"] / n
                                    rec["foot_offset_mean_rel"] = rec["foot_offset_sum"] / n
    
                                    del rec["arm_inward_sum"]
                                    del rec["fist_height_sum"]
                                    del rec["foot_offset_sum"]
                                    del rec["n"]
    
                                    rec["done"] = True
                                    leg_is_on_ground[side] = False
                                    current_status_event = f"{side.upper()} PUSH-OFF"
                                break
    
                # ui text display
                cv2.rectangle(display_frame, (0, 0), (280, 100), (20, 20, 20), -1)
                cv2.putText(display_frame, f"STEPS: {right_step_count + left_step_count}", (15, 35), 1, 1.8, (255, 255, 255), 2)
                cv2.putText(display_frame, f"CADENCE: {int(avg_cadence)}", (15, 65), 1, 1.2, (0, 255, 0), 2)
                cv2.putText(display_frame, f"STATUS: {current_status_event}", (15, 95), 1, 1.0, (0, 255, 255), 1)
    
                for side_ui, pos, state in [
                    ("LEFT", (10, frame_height - 20), leg_is_on_ground["Left"]),
                    ("RIGHT", (frame_width - 130, frame_height - 20), leg_is_on_ground["Right"])
                ]:
                    color = (0, 255, 0) if state else (0, 0, 255)
                    cv2.rectangle(display_frame, (pos[0] - 10, pos[1] - 40), (pos[0] + 120, pos[1] + 10), (0, 0, 0), -1)
                    cv2.putText(display_frame, side_ui, (pos[0], pos[1] - 20), 1, 1.2, color, 2)
                    cv2.putText(display_frame, "CONTACT" if state else "FLIGHT", (pos[0], pos[1]), 1, 0.9, (255, 255, 255), 1)
    
                cv2.imshow("Running analysis", display_frame)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break

    video_capture.release()
    cv2.destroyAllWindows()
    return all_step_metrics_storage

In [27]:
# baseline metric score intervals
def get_metric_score_front(metric, val):
    if metric == "cadence":
        if val > 175: return 3
        if val > 168: return 2
        return 1

    if metric == "gct":
        if val < 265: return 3
        if val < 283: return 2
        return 1

    if metric == "hip_drop":
        if val < 5: return 3
        if val < 7: return 2
        if val < 10: return 1
        return 0

    if metric == "shoulder_drop":
        if val < 5: return 3
        if val < 7: return 2
        if val < 10: return 1
        return 0

    if metric == "foot_offset_mean_rel":
        a = abs(val)
        if a <= 0.12: return 3
        if a <= 0.20: return 2
        if a <= 0.28: return 1
        return 0

    if metric == "arm_inward_deg":
        if val < 15: return 3
        if val < 30: return 2
        if val < 45: return 1
        return 0

    if metric == "fist_height_rel":
        if val <= 0.10: return 3
        if val <= 0.20: return 2
        if val <= 0.30: return 1
        return 0

    return 0

In [28]:
# csv storing 
fieldnames = [
    'rep_number', 'frame_index', 'file_name', 'side',

    'cadence_value_spm', 'gct_value_ms',
    'hip_drop_max_deg', 'shoulder_drop_max_deg',
    'foot_offset_mean_rel', 'arm_inward_mean_deg', 'fist_height_mean_rel',

    'cadence_score', 'gct_score',
    'hip_drop_score', 'shoulder_drop_score',
    'foot_offset_score', 'arm_inward_score', 'fist_height_score',

    # ml features
    'n_ankle_x', 'n_ankle_y', 'n_knee_x', 'n_knee_y',
    'n_knee_to_hip_x', 'n_ankle_to_hip_x',
    'n_shoulder_x', 'n_elbow_x', 'n_wrist_x'
]

def save_steps_to_csv_front(steps_list, video_path, output_file='front_view_dataset.csv'):
    # file setup
    file_exists = os.path.isfile(output_file)

    with open(output_file, 'a', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()

        line_count = 0
        if file_exists:
            with open(output_file, 'r') as f:
                line_count = sum(1 for _ in f) - 1

        for i, m in enumerate(steps_list, start=max(0, line_count) + 1):
            if not m.get('done'):
                continue

            cad = float(m.get('cadence', 0))
            gct = float(m.get('gct', 0))

            hip_drop = float(m.get('hip_drop_max_deg', 0))
            sh_drop  = float(m.get('shoulder_drop_max_deg', 0))

            foot_off = float(m.get('foot_offset_mean_rel', 0))
            arm_in   = float(m.get('arm_inward_mean_deg', 0))
            fist_h   = float(m.get('fist_height_mean_rel', 0))

            # row data mapping
            row = {
                'rep_number': i,
                'frame_index': int(m.get('start_frame', 0)),
                'file_name': video_path,
                'side': m.get('side', ''),

                'cadence_value_spm': round(cad, 2),
                'gct_value_ms': int(gct),

                'hip_drop_max_deg': round(hip_drop, 2),
                'shoulder_drop_max_deg': round(sh_drop, 2),
                'foot_offset_mean_rel': round(foot_off, 4),
                'arm_inward_mean_deg': round(arm_in, 2),
                'fist_height_mean_rel': round(fist_h, 4),

                'cadence_score': get_metric_score_front('cadence', cad),
                'gct_score': get_metric_score_front('gct', gct),
                'hip_drop_score': get_metric_score_front('hip_drop', hip_drop),
                'shoulder_drop_score': get_metric_score_front('shoulder_drop', sh_drop),
                'foot_offset_score': get_metric_score_front('foot_offset_mean_rel', foot_off),
                'arm_inward_score': get_metric_score_front('arm_inward_deg', arm_in),
                'fist_height_score': get_metric_score_front('fist_height_rel', fist_h),

                # ml feature mapping
                'n_ankle_x': m.get('n_ankle_x'),
                'n_ankle_y': m.get('n_ankle_y'),
                'n_knee_x': m.get('n_knee_x'),
                'n_knee_y': m.get('n_knee_y'),
                'n_knee_to_hip_x': m.get('n_knee_to_hip_x'),
                'n_ankle_to_hip_x': m.get('n_ankle_to_hip_x'),
                'n_shoulder_x': m.get('n_shoulder_x'),
                'n_elbow_x': m.get('n_elbow_x'),
                'n_wrist_x': m.get('n_wrist_x'),

                # score calculation - labels (FRONT VIEW)
                'cadence_score': get_metric_score_front("cadence", m['cadence']),
                'gct_score': get_metric_score_front("gct", m['gct']),
                'hip_drop_score': get_metric_score_front("hip_drop", m['hip_drop_max_deg']),
                'shoulder_drop_score': get_metric_score_front("shoulder_drop", m['shoulder_drop_max_deg']),
                'foot_offset_score': get_metric_score_front("foot_offset_mean_rel", m['foot_offset_mean_rel']),
                'arm_inward_score': get_metric_score_front("arm_inward_deg", m['arm_inward_mean_deg']),
                'fist_height_score': get_metric_score_front("fist_height_rel", m['fist_height_mean_rel']),

            }

            writer.writerow(row)

    print(f"Uspješno spremljeno {sum(1 for s in steps_list if s.get('done'))} koraka.")


In [29]:
# dataset creation
paths = ['./Videos/tr1.mp4']

for video_path in paths:
    # data processing
    captured_data = analyze_video(video_path)

    # dataset saving and scoring
    if captured_data:
        save_steps_to_csv_front(captured_data, video_path)

print("Dataset ažuriran.")

I0000 00:00:1769280977.301250  405857 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
W0000 00:00:1769280977.451015  492709 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769280977.463824  492713 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/opt/miniconda3/envs/runai/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Uspješno spremljeno 9 koraka.
Dataset ažuriran.


In [30]:
# model training and evaluating (FRONT VIEW)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

# loading data
df = pd.read_csv('front_view_dataset.csv')

# feature and target definition
X_features = [
    'cadence_value_spm', 'gct_value_ms',
    'hip_drop_max_deg', 'shoulder_drop_max_deg',
    'foot_offset_mean_rel', 'arm_inward_mean_deg', 'fist_height_mean_rel',
    'n_ankle_x', 'n_ankle_y', 'n_knee_x', 'n_knee_y',
    'n_knee_to_hip_x', 'n_ankle_to_hip_x',
    'n_shoulder_x', 'n_elbow_x', 'n_wrist_x'
]

target_scores = [
    'cadence_score', 'gct_score',
    'hip_drop_score', 'shoulder_drop_score',
    'foot_offset_score', 'arm_inward_score', 'fist_height_score'
]

# data cleaning
df_clean = df[df['cadence_value_spm'] > 0].dropna()
X = df_clean[X_features]
Y = df_clean[target_scores]

# train test split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

# model training and evaluation
print(f"═" * 40)
print(f"{'METRIKA':<22} | {'TOČNOST':<10}")
print(f"─" * 40)

models = {}
for target in target_scores:
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, Y_train[target])

    y_pred = clf.predict(X_test)
    acc = accuracy_score(Y_test[target], y_pred)

    models[target] = clf
    print(f"{target:<22} | {acc:.2%}")

print(f"═" * 40)

# sample prediction
sample_idx = 0
real_values = Y_test.iloc[sample_idx].values
predicted_values = [int(models[t].predict(X_test.iloc[[sample_idx]])[0]) for t in target_scores]


print("\nprovjera na jednom koraku:")
print(f"stvarne ocjene:    {real_values}")
print(f"predviđene ocjene: {predicted_values}")


════════════════════════════════════════
METRIKA                | TOČNOST   
────────────────────────────────────────
cadence_score          | 97.06%
gct_score              | 100.00%
hip_drop_score         | 97.06%
shoulder_drop_score    | 94.12%
foot_offset_score      | 97.06%
arm_inward_score       | 94.12%
fist_height_score      | 100.00%
════════════════════════════════════════

provjera na jednom koraku:
stvarne ocjene:    [1 3 3 3 0 2 3]
predviđene ocjene: [1, 3, 3, 3, 0, 2, 3]


In [31]:
# model testing (FRONT VIEW)
def test_on_new_video_front(video_path, trained_models, features_list):
    print(f"--- analiza videa: {video_path} ---")

    # video analysis and data extraction
    raw_steps_data = analyze_video(video_path)
    if not raw_steps_data:
        print("nema detektiranih koraka.")
        return

    # dataframe preparation
    test_df = pd.DataFrame(raw_steps_data)
    test_df = test_df[test_df["done"] == True].copy()
    if test_df.empty:
        print("nema dovršenih (done) koraka.")
        return

    # column mapping for model compatibility (front view)
    column_mapping_for_model = {
        "cadence": "cadence_value_spm",
        "gct": "gct_value_ms"
    }
    model_df = test_df.rename(columns=column_mapping_for_model)

    # target mapping for math score comparison (front view)
    target_to_math_key = {
        "cadence_score": "cadence",
        "gct_score": "gct",
        "hip_drop_score": "hip_drop_max_deg",
        "shoulder_drop_score": "shoulder_drop_max_deg",
        "foot_offset_score": "foot_offset_mean_rel",
        "arm_inward_score": "arm_inward_mean_deg",
        "fist_height_score": "fist_height_mean_rel"
    }

    print(f"\n{'metrika':<22} | {'predviđanje':<12} | {'matematički izračun'}")
    print("-" * 62)

    # evaluation loop
    X_input = model_df[features_list]

    for target, model in trained_models.items():
        # model prediction
        ai_preds = model.predict(X_input)
        avg_ai_score = int(round(np.mean(ai_preds)))

        # math score comparison
        math_key = target_to_math_key.get(target)
        if math_key in test_df.columns:
            avg_val = float(test_df[math_key].mean())

            # score by metric name used in get_metric_score_front
            if math_key == "cadence":
                your_score = get_metric_score_front("cadence", avg_val)
            elif math_key == "gct":
                your_score = get_metric_score_front("gct", avg_val)
            elif math_key == "hip_drop_max_deg":
                your_score = get_metric_score_front("hip_drop", avg_val)
            elif math_key == "shoulder_drop_max_deg":
                your_score = get_metric_score_front("shoulder_drop", avg_val)
            elif math_key == "foot_offset_mean_rel":
                your_score = get_metric_score_front("foot_offset_mean_rel", avg_val)
            elif math_key == "arm_inward_mean_deg":
                your_score = get_metric_score_front("arm_inward_deg", avg_val)
            elif math_key == "fist_height_mean_rel":
                your_score = get_metric_score_front("fist_height_rel", avg_val)
            else:
                your_score = "N/A"
        else:
            your_score = "N/A"

        print(f"{target:<22} | {avg_ai_score:<12} | {your_score}")


# pokretanje (FRONT VIEW)
novi_video = "./Videos/tr1.mp4"
test_on_new_video_front(novi_video, models, X_features)


--- analiza videa: ./Videos/tr1.mp4 ---


I0000 00:00:1769281004.494258  405857 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
W0000 00:00:1769281004.600403  493560 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769281004.615046  493560 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/opt/miniconda3/envs/runai/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '



metrika                | predviđanje  | matematički izračun
--------------------------------------------------------------
cadence_score          | 2            | 1
gct_score              | 3            | 3
hip_drop_score         | 1            | 1
shoulder_drop_score    | 2            | 2
foot_offset_score      | 2            | 3
arm_inward_score       | 3            | 3
fist_height_score      | 3            | 3
